# SP-8: Player Performance Analysis

**Runs AFTER SP-7** to capture all features from the pipeline.

Identifies winning players based on long-term profitability.

**Purpose:** Instead of labeling actions as "good" based on single-hand outcomes,
we identify players who are profitable over many hands. Their actions become
the training labels for the policy model (SP-9).

**Why this matters:**
- Single hand outcomes have high variance (AA can lose to 72)
- Winning players make good decisions ON AVERAGE
- Learning from winners teaches real poker strategy, not luck

**Pipeline Position:**
```
SP-2 → SP-3 → SP-4 → SP-5 → SP-6 → SP-7 → [SP-8] → SP-9
                                            ↑
                                      (this notebook)
```

**Output:** SP-7 features + player performance metrics

In [ ]:
# SP-8: Player Performance Analysis
# Identifies winning players for policy training labels
# INPUT: SP-7 output (has all features from pipeline)
# OUTPUT: SP-7 features + player performance metrics

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, StringType
import time
import json

spark = SparkSession.builder.getOrCreate()

print("=" * 80)
print("SP-8: Player Performance Analysis")
print("=" * 80)
print("\nPipeline Position: SP-7 → [SP-8] → SP-9")
print("\nPurpose: Identify winning players for policy model training")
print("Method: Calculate profit per 100 hands for each player")
print("Output: SP-7 features + is_winning_player flag")
start_time = time.time()

In [ ]:
# Configuration
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# ============================================================================
# INPUT: SP-7 output (has ALL features from pipeline)
# This includes: SP-2 base features, SP-3 opponent predictions, SP-4 advanced,
#                SP-6 policy features, SP-7 action labels
# ============================================================================
SP7_INPUT_PATH = UC_VOLUME_DIR + 'processed/sp7_action_labels'

# Output: Enhanced features with player performance
OUTPUT_PATH = UC_VOLUME_DIR + 'processed/sp8_player_performance'

# Player stats output (for analysis)
PLAYER_STATS_PATH = UC_VOLUME_DIR + 'processed/sp8_player_stats'

# ============================================================================
# WINNING PLAYER THRESHOLDS
# ============================================================================
# Profit per 100 hands threshold (in BB)
# - Recreational players: typically -10 to -30 BB/100
# - Break-even players: around 0 BB/100
# - Winning players: +5 to +15 BB/100
# - Strong winners: +15 to +30 BB/100
# - Elite players: +30+ BB/100
WINNING_THRESHOLD_BB_PER_100 = 5.0  # At least +5 BB per 100 hands

# Minimum hands to be considered (avoid small sample luck)
# NOTE: Lower this if you have limited data per player
MIN_HANDS_FOR_EVALUATION = 10  # Reduced from 50 to work with smaller datasets

# ============================================================================
# DEBUG MODE - Read from pipeline config
# ============================================================================
CONFIG_PATH = '/Volumes/pokerml/default/data/pipeline_config.json'

try:
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    DEBUG_MODE = config.get('debug_mode', True)
    MAX_ROWS = config.get('max_rows', 10000)
    print(f"   Loaded config: DEBUG_MODE={DEBUG_MODE}, MAX_ROWS={MAX_ROWS:,}")
except FileNotFoundError:
    DEBUG_MODE = True
    MAX_ROWS = 10000
    print(f"   Config not found, using defaults")

print(f"\nInput: {SP7_INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"\nWinning threshold: >= {WINNING_THRESHOLD_BB_PER_100} BB/100 hands")
print(f"Minimum hands required: {MIN_HANDS_FOR_EVALUATION}")

In [ ]:
# Load SP-7 data (has ALL features from pipeline)
print("\n[1/7] Loading SP-7 features...")

df = spark.read.parquet(SP7_INPUT_PATH)
initial_count = df.count()
print(f"   Loaded {initial_count:,} rows")
print(f"   Columns: {len(df.columns)}")

# Show what features we inherited from the pipeline
print("\n   Features inherited from pipeline:")
print(f"   Total columns: {len(df.columns)}")

# Verify required columns
required_cols = ['hand_id', 'actor', 'target_profit_bb']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"   ✓ Required columns present: {required_cols}")

# ============================================================================
# TRACE PATTERN: Select a hand_id for pipeline validation
# ============================================================================
TRACE_HAND_ID = df.select('hand_id').first()['hand_id']
print(f"\n[TRACE] Selected TRACE_HAND_ID: {TRACE_HAND_ID}")

# TRACE: Show raw input for trace hand
print(f"\n[TRACE] RAW INPUT from SP-7 for {TRACE_HAND_ID}:")
trace_cols = ['hand_id', 'actor', 'street', 'action_type', 'target_profit_bb', 'hand_equity']
trace_cols = [c for c in trace_cols if c in df.columns]
df.filter(F.col('hand_id') == TRACE_HAND_ID).select(trace_cols).show(20, truncate=False)

In [ ]:
# Calculate player-level performance
print("\n[2/7] Calculating player performance metrics...")

# Get one profit value per (hand_id, actor) - avoid counting same hand multiple times
# Take the profit from the LAST action in the hand (most complete)
player_hand_profit = df.groupBy('hand_id', 'actor').agg(
    F.last('target_profit_bb').alias('hand_profit_bb')
)

# Aggregate to player level
player_stats = player_hand_profit.groupBy('actor').agg(
    F.count('hand_id').alias('total_hands'),
    F.sum('hand_profit_bb').alias('total_profit_bb'),
    F.avg('hand_profit_bb').alias('avg_profit_per_hand_bb'),
    F.stddev('hand_profit_bb').alias('profit_stddev'),
    F.sum(F.when(F.col('hand_profit_bb') > 0, 1).otherwise(0)).alias('winning_hands'),
    F.sum(F.when(F.col('hand_profit_bb') < 0, 1).otherwise(0)).alias('losing_hands'),
    F.sum(F.when(F.col('hand_profit_bb') == 0, 1).otherwise(0)).alias('breakeven_hands')
)

# Calculate profit per 100 hands (standard poker metric)
player_stats = player_stats.withColumn(
    'profit_per_100_hands',
    (F.col('total_profit_bb') / F.col('total_hands')) * 100
)

# Calculate win rate
player_stats = player_stats.withColumn(
    'win_rate',
    F.col('winning_hands') / F.col('total_hands')
)

print(f"   Unique players: {player_stats.count():,}")

# Show distribution of profit per 100 hands
print("\n   Profit per 100 hands distribution:")
player_stats.select('profit_per_100_hands').summary('min', '25%', '50%', '75%', 'max', 'mean').show()

# ============================================================================
# TRACE: Show stats for actors in TRACE_HAND_ID
# ============================================================================
print(f"\n[TRACE] Player stats for actors in {TRACE_HAND_ID}:")
trace_actors = df.filter(F.col('hand_id') == TRACE_HAND_ID).select('actor').distinct().collect()
trace_actor_list = [r['actor'] for r in trace_actors]
player_stats.filter(F.col('actor').isin(trace_actor_list)).show(truncate=False)

In [ ]:
# Classify players using DECILES (10 groups based on performance)
print("\n[3/7] Classifying players into performance deciles...")

# ============================================================================
# DECILE-BASED CLASSIFICATION
# ============================================================================
# Instead of arbitrary thresholds, we rank players by profit_per_100_hands
# and assign them to deciles (1-10, where 10 = top 10% performers)
#
# This approach:
# 1. Adapts to the actual distribution of player skill in the data
# 2. Guarantees exactly 10% of players in the top group
# 3. Makes SP-9 training more selective (only learns from elite players)
# ============================================================================

# First, filter to players with enough hands
player_stats = player_stats.withColumn(
    'has_enough_hands',
    F.col('total_hands') >= MIN_HANDS_FOR_EVALUATION
)

# Count players with enough hands
players_with_enough = player_stats.filter(F.col('has_enough_hands')).count()
players_without_enough = player_stats.filter(~F.col('has_enough_hands')).count()
print(f"   Players with >= {MIN_HANDS_FOR_EVALUATION} hands: {players_with_enough:,}")
print(f"   Players with insufficient hands: {players_without_enough:,}")

# ============================================================================
# CALCULATE DECILES ONLY FOR PLAYERS WITH ENOUGH HANDS
# ============================================================================
# Split into two groups, calculate deciles for qualified players, then union back

# Players with enough hands - calculate deciles
qualified_players = player_stats.filter(F.col('has_enough_hands'))

window_spec = Window.orderBy(F.col('profit_per_100_hands'))

qualified_players = qualified_players.withColumn(
    'performance_decile',
    F.ntile(10).over(window_spec)
)

# Players without enough hands - set decile to 0
unqualified_players = player_stats.filter(~F.col('has_enough_hands')).withColumn(
    'performance_decile',
    F.lit(0)
)

# Union back together
player_stats = qualified_players.unionByName(unqualified_players)

# Show decile distribution
print("\n   Performance decile distribution:")
decile_stats = player_stats.filter(F.col('has_enough_hands')).groupBy('performance_decile').agg(
    F.count('*').alias('player_count'),
    F.min('profit_per_100_hands').alias('min_bb_100'),
    F.max('profit_per_100_hands').alias('max_bb_100'),
    F.avg('profit_per_100_hands').alias('avg_bb_100'),
    F.avg('total_hands').alias('avg_hands')
).orderBy('performance_decile')

decile_stats.show(10, truncate=False)

# Get the threshold for top decile (decile 10)
top_decile_threshold = player_stats.filter(
    F.col('performance_decile') == 10
).agg(F.min('profit_per_100_hands')).collect()[0][0]

print(f"\n   Top decile (10) threshold: >= {top_decile_threshold:.2f} BB/100")

# ============================================================================
# SET is_winning_player = 1 ONLY FOR TOP DECILE
# ============================================================================
player_stats = player_stats.withColumn(
    'is_winning_player',
    F.when(F.col('performance_decile') == 10, 1).otherwise(0)
)

# Create descriptive player categories based on deciles
player_stats = player_stats.withColumn(
    'player_category',
    F.when(~F.col('has_enough_hands'), 'insufficient_data')
    .when(F.col('performance_decile') == 10, 'elite (top 10%)')
    .when(F.col('performance_decile') == 9, 'strong (top 20%)')
    .when(F.col('performance_decile') >= 7, 'above_average (top 40%)')
    .when(F.col('performance_decile') >= 5, 'average (40-60%)')
    .when(F.col('performance_decile') >= 3, 'below_average (bottom 40%)')
    .otherwise('weak (bottom 20%)')
)

# Show player category distribution
print("\n   Player category distribution:")
player_stats.groupBy('player_category').agg(
    F.count('*').alias('count'),
    F.avg('profit_per_100_hands').alias('avg_bb_per_100'),
    F.avg('total_hands').alias('avg_hands')
).orderBy('avg_bb_per_100', ascending=False).show()

# Count winning (top decile) vs non-winning
winning_count = player_stats.filter(F.col('is_winning_player') == 1).count()
total_players = player_stats.count()
print(f"\n   Elite players (top 10%): {winning_count:,} ({100*winning_count/total_players:.1f}%)")
print(f"   Other players: {total_players - winning_count:,} ({100*(total_players-winning_count)/total_players:.1f}%)")
print(f"\n   SP-9 will train ONLY on actions from elite (top 10%) players")

In [ ]:
# Show top winners and their stats
print("\n[DEBUG] Top 10 winning players:")
player_stats.filter(
    F.col('has_enough_hands')
).orderBy('profit_per_100_hands', ascending=False).select(
    'actor', 'total_hands', 'total_profit_bb', 'profit_per_100_hands', 
    'win_rate', 'player_category'
).show(10, truncate=False)

print("\n[DEBUG] Bottom 10 players (biggest losers):")
player_stats.filter(
    F.col('has_enough_hands')
).orderBy('profit_per_100_hands', ascending=True).select(
    'actor', 'total_hands', 'total_profit_bb', 'profit_per_100_hands', 
    'win_rate', 'player_category'
).show(10, truncate=False)

In [ ]:
# Join player performance back to action data
print("\n[4/7] Joining player performance to action data...")

# Select columns to join (now including performance_decile)
player_lookup = player_stats.select(
    'actor',
    'total_hands',
    'profit_per_100_hands',
    'performance_decile',
    'is_winning_player',
    'player_category',
    'win_rate'
)

# Join to original data
rows_before = df.count()
df_with_performance = df.join(player_lookup, on='actor', how='left')
rows_after = df_with_performance.count()

print(f"   Before join: {rows_before:,} rows")
print(f"   After join: {rows_after:,} rows")

if rows_after != rows_before:
    print(f"   *** WARNING: Row count changed! ***")
else:
    print(f"   ✓ Row count preserved")

# Fill nulls for players without enough hands
df_with_performance = df_with_performance.fillna({
    'is_winning_player': 0,
    'performance_decile': 0,
    'player_category': 'insufficient_data',
    'profit_per_100_hands': 0.0,
    'win_rate': 0.0
})

# Show distribution of actions by player category
print("\n   Actions by player category:")
df_with_performance.groupBy('player_category').agg(
    F.count('*').alias('action_count'),
    F.countDistinct('hand_id').alias('hand_count'),
    F.countDistinct('actor').alias('player_count')
).orderBy('action_count', ascending=False).show()

# Show actions by decile
print("\n   Actions by performance decile:")
df_with_performance.groupBy('performance_decile').agg(
    F.count('*').alias('action_count'),
    F.countDistinct('actor').alias('player_count')
).orderBy('performance_decile').show(11)

In [ ]:
# TRACE: Show performance for TRACE_HAND_ID actors
print(f"\n[TRACE] Player performance for actors in {TRACE_HAND_ID}:")

trace_actors = df_with_performance.filter(
    F.col('hand_id') == TRACE_HAND_ID
).select('actor').distinct().collect()

trace_actor_list = [r['actor'] for r in trace_actors]
print(f"   Actors in hand: {trace_actor_list}")

player_stats.filter(
    F.col('actor').isin(trace_actor_list)
).select(
    'actor', 'total_hands', 'profit_per_100_hands', 
    'is_winning_player', 'player_category'
).show(truncate=False)

In [ ]:
# Save output
print("\n[5/7] Saving output...")

# Verify performance_decile column exists before saving
print("\n   Verifying columns before save:")
print(f"   - performance_decile in player_lookup: {'performance_decile' in [c for c in player_lookup.columns]}")
print(f"   - performance_decile in df_with_performance: {'performance_decile' in [c for c in df_with_performance.columns]}")

# Show sample of performance_decile values
print("\n   Sample performance_decile values:")
df_with_performance.select('actor', 'performance_decile', 'player_category').distinct().show(10)

# Cast performance_decile to integer to ensure proper type
df_with_performance = df_with_performance.withColumn(
    'performance_decile',
    F.col('performance_decile').cast('int')
)

# Save enhanced data with player performance
df_with_performance.write.mode('overwrite').parquet(OUTPUT_PATH)

# Also save player stats separately for reference
player_stats.write.mode('overwrite').parquet(PLAYER_STATS_PATH)

final_count = df_with_performance.count()
final_cols = len(df_with_performance.columns)

print(f"\n   Saved: {OUTPUT_PATH}")
print(f"   Saved: {PLAYER_STATS_PATH}")
print(f"   Dataset: {final_count:,} rows, {final_cols} columns")

# Verify the saved file has performance_decile
print("\n   Verifying saved output:")
verify_df = spark.read.parquet(OUTPUT_PATH)
print(f"   Columns in saved file: {len(verify_df.columns)}")
print(f"   performance_decile exists: {'performance_decile' in verify_df.columns}")
if 'performance_decile' in verify_df.columns:
    verify_df.groupBy('performance_decile').count().orderBy('performance_decile').show(11)

# ============================================================================
# TRACE: Verify output for TRACE_HAND_ID
# ============================================================================
print(f"\n[TRACE] FINAL OUTPUT for {TRACE_HAND_ID}:")
df_with_performance.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type', 
    'performance_decile', 'is_winning_player', 'player_category', 'profit_per_100_hands'
).show(20, truncate=False)

In [ ]:
# ============================================================================
# [6/7] VISUALIZATIONS: Top Decile vs Bottom Decile Analysis
# ============================================================================
print("\n[6/7] Creating comprehensive visualizations...")

import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd
import numpy as np

# ============================================================================
# VISUAL THEME: Force white background (overrides Databricks dark mode)
# ============================================================================
# Set default renderer for Databricks
pio.templates.default = 'plotly_white'

# Define colors as constants
ELITE_GREEN = '#27ae60'   # Green for Elite/Winners
WEAK_RED = '#e74c3c'      # Red for Weak/Losers

print(f"   Using colors: Elite={ELITE_GREEN} (green), Weakest={WEAK_RED} (red)")

# ============================================================================
# FILTER TO EXTREME GROUPS: Top 10% vs Bottom 10%
# ============================================================================
print("\n   Filtering to extreme deciles for clearer comparison...")

# Create comparison groups based on deciles
df_extremes = df_with_performance.withColumn(
    'comparison_group',
    F.when(F.col('performance_decile') == 10, 'Elite (Top 10%)')
    .when(F.col('performance_decile') == 1, 'Weakest (Bottom 10%)')
    .otherwise(None)
).filter(F.col('comparison_group').isNotNull())

extreme_counts = df_extremes.groupBy('comparison_group').count().toPandas()
print("\n   Comparison groups:")
for _, row in extreme_counts.iterrows():
    print(f"      {row['comparison_group']}: {row['count']:,} actions")

# ============================================================================
# VISUALIZATION 1: Action Distribution - Top vs Bottom Decile
# ============================================================================
print("\n   Creating action distribution chart (Top 10% vs Bottom 10%)...")

action_extreme = df_extremes.groupBy('comparison_group', 'action_type').agg(
    F.count('*').alias('count')
).toPandas()

action_extreme['total'] = action_extreme.groupby('comparison_group')['count'].transform('sum')
action_extreme['pct'] = action_extreme['count'] / action_extreme['total'] * 100

# Sort action types for consistent ordering
action_order = ['fold', 'call_or_check', 'bet_or_raise_to']
action_extreme['action_type'] = pd.Categorical(action_extreme['action_type'], categories=action_order, ordered=True)
action_extreme = action_extreme.sort_values(['comparison_group', 'action_type'])

# Get data for each group
elite_data = action_extreme[action_extreme['comparison_group'] == 'Elite (Top 10%)'].sort_values('action_type')
weak_data = action_extreme[action_extreme['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('action_type')

# Create figure with explicit bar colors
fig1 = go.Figure()

# Add Elite bars (GREEN)
fig1.add_trace(go.Bar(
    name='Elite (Top 10%)',
    x=elite_data['action_type'].tolist(),
    y=elite_data['pct'].tolist(),
    marker=dict(color=ELITE_GREEN),
    text=[f'{v:.1f}%' for v in elite_data['pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

# Add Weakest bars (RED)
fig1.add_trace(go.Bar(
    name='Weakest (Bottom 10%)',
    x=weak_data['action_type'].tolist(),
    y=weak_data['pct'].tolist(),
    marker=dict(color=WEAK_RED),
    text=[f'{v:.1f}%' for v in weak_data['pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

fig1.update_layout(
    title=dict(
        text='<b>Action Distribution: Top 10% vs Bottom 10% Players</b><br><sup>Green=Elite, Red=Weakest</sup>',
        font=dict(color='black', size=16)
    ),
    xaxis_title='Action Type',
    yaxis_title='Percentage of Actions (%)',
    barmode='group',
    height=500,
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(
        bgcolor='white',
        font=dict(color='black', size=12),
        title=dict(text='Player Group', font=dict(color='black'))
    ),
    xaxis=dict(
        showgrid=True, gridcolor='lightgray', linecolor='black',
        tickfont=dict(color='black', size=12)
    ),
    yaxis=dict(
        showgrid=True, gridcolor='lightgray', linecolor='black',
        tickfont=dict(color='black', size=12),
        range=[0, max(action_extreme['pct']) * 1.15]  # Add headroom for labels
    )
)

fig1.show()

In [ ]:
# ============================================================================
# VISUALIZATION 2: Street-by-Street Aggression Analysis (Top vs Bottom Decile)
# ============================================================================
print("\n   Creating street aggression chart...")

street_agg = df_extremes.groupBy('comparison_group', 'street').agg(
    F.sum(F.when(F.col('action_type') == 'bet_or_raise_to', 1).otherwise(0)).alias('raises'),
    F.sum(F.when(F.col('action_type') == 'call_or_check', 1).otherwise(0)).alias('calls'),
    F.sum(F.when(F.col('action_type') == 'fold', 1).otherwise(0)).alias('folds'),
    F.count('*').alias('total_actions')
).toPandas()

street_agg['aggression_pct'] = street_agg['raises'] / (street_agg['calls'] + street_agg['raises'] + 0.001) * 100

# Order streets properly
street_order = ['preflop', 'flop', 'turn', 'river']
street_agg['street'] = pd.Categorical(street_agg['street'], categories=street_order, ordered=True)
street_agg = street_agg.sort_values('street')

# Get data for each group
elite_street = street_agg[street_agg['comparison_group'] == 'Elite (Top 10%)'].sort_values('street')
weak_street = street_agg[street_agg['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('street')

fig2 = go.Figure()

# Add Elite bars (GREEN)
fig2.add_trace(go.Bar(
    name='Elite (Top 10%)',
    x=elite_street['street'].tolist(),
    y=elite_street['aggression_pct'].tolist(),
    marker=dict(color=ELITE_GREEN),
    text=[f'{v:.1f}%' for v in elite_street['aggression_pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

# Add Weakest bars (RED)
fig2.add_trace(go.Bar(
    name='Weakest (Bottom 10%)',
    x=weak_street['street'].tolist(),
    y=weak_street['aggression_pct'].tolist(),
    marker=dict(color=WEAK_RED),
    text=[f'{v:.1f}%' for v in weak_street['aggression_pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

fig2.update_layout(
    title=dict(
        text='<b>Aggression by Street: Top 10% vs Bottom 10%</b><br><sup>Aggression = Raises / (Calls + Raises)</sup>',
        font=dict(color='black', size=16)
    ),
    xaxis_title='Street',
    yaxis_title='Aggression %',
    barmode='group',
    height=500,
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(bgcolor='white', font=dict(color='black', size=12)),
    xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12)),
    yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
)

fig2.show()

In [ ]:
# ============================================================================
# VISUALIZATION 3: Position-Based Play Analysis (Top vs Bottom Decile)
# ============================================================================
print("\n   Creating position analysis chart...")

position_stats = df_extremes.groupBy('comparison_group', 'position_from_button').agg(
    F.sum(F.when(F.col('action_type') == 'bet_or_raise_to', 1).otherwise(0)).alias('raises'),
    F.count('*').alias('total_actions')
).toPandas()

position_stats['raise_pct'] = position_stats['raises'] / position_stats['total_actions'] * 100

# Map position numbers to names
pos_map = {0: 'BTN', 1: 'SB', 2: 'BB', 3: 'UTG', 4: 'MP', 5: 'CO', 6: 'HJ'}
position_stats['position_name'] = position_stats['position_from_button'].map(pos_map).fillna('Other')

# Get data for each group
elite_pos = position_stats[position_stats['comparison_group'] == 'Elite (Top 10%)'].sort_values('position_from_button')
weak_pos = position_stats[position_stats['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('position_from_button')

fig3 = go.Figure()

# Add Elite bars (GREEN)
fig3.add_trace(go.Bar(
    name='Elite (Top 10%)',
    x=elite_pos['position_name'].tolist(),
    y=elite_pos['raise_pct'].tolist(),
    marker=dict(color=ELITE_GREEN),
    text=[f'{v:.1f}%' for v in elite_pos['raise_pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

# Add Weakest bars (RED)
fig3.add_trace(go.Bar(
    name='Weakest (Bottom 10%)',
    x=weak_pos['position_name'].tolist(),
    y=weak_pos['raise_pct'].tolist(),
    marker=dict(color=WEAK_RED),
    text=[f'{v:.1f}%' for v in weak_pos['raise_pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

fig3.update_layout(
    title=dict(
        text='<b>Raise % by Position: Top 10% vs Bottom 10%</b><br><sup>How aggressive are players in each position?</sup>',
        font=dict(color='black', size=16)
    ),
    xaxis_title='Position',
    yaxis_title='Raise %',
    barmode='group',
    height=500,
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(bgcolor='white', font=dict(color='black', size=12)),
    xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12)),
    yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
)

fig3.show()

In [ ]:
# ============================================================================
# VISUALIZATION 4: Hand Strength vs Action Analysis (Top vs Bottom Decile)
# ============================================================================
print("\n   Creating hand strength analysis chart...")

# Bin hand equity into categories
hand_strength_stats = df_extremes.withColumn(
    'equity_bucket',
    F.when(F.col('hand_equity') < 0.2, '0-20% (Weak)')
    .when(F.col('hand_equity') < 0.4, '20-40% (Below Avg)')
    .when(F.col('hand_equity') < 0.6, '40-60% (Average)')
    .when(F.col('hand_equity') < 0.8, '60-80% (Strong)')
    .otherwise('80-100% (Premium)')
).groupBy('comparison_group', 'equity_bucket', 'action_type').agg(
    F.count('*').alias('count')
).toPandas()

# Calculate percentage within each group/equity bucket
hand_strength_stats['total'] = hand_strength_stats.groupby(['comparison_group', 'equity_bucket'])['count'].transform('sum')
hand_strength_stats['pct'] = hand_strength_stats['count'] / hand_strength_stats['total'] * 100

# Focus on raises
raise_by_equity = hand_strength_stats[hand_strength_stats['action_type'] == 'bet_or_raise_to'].copy()

bucket_order = ['0-20% (Weak)', '20-40% (Below Avg)', '40-60% (Average)', '60-80% (Strong)', '80-100% (Premium)']
raise_by_equity['equity_bucket'] = pd.Categorical(raise_by_equity['equity_bucket'], categories=bucket_order, ordered=True)
raise_by_equity = raise_by_equity.sort_values('equity_bucket')

# Get data for each group
elite_equity = raise_by_equity[raise_by_equity['comparison_group'] == 'Elite (Top 10%)'].sort_values('equity_bucket')
weak_equity = raise_by_equity[raise_by_equity['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('equity_bucket')

fig4 = go.Figure()

# Add Elite bars (GREEN)
fig4.add_trace(go.Bar(
    name='Elite (Top 10%)',
    x=elite_equity['equity_bucket'].tolist(),
    y=elite_equity['pct'].tolist(),
    marker=dict(color=ELITE_GREEN),
    text=[f'{v:.1f}%' for v in elite_equity['pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

# Add Weakest bars (RED)
fig4.add_trace(go.Bar(
    name='Weakest (Bottom 10%)',
    x=weak_equity['equity_bucket'].tolist(),
    y=weak_equity['pct'].tolist(),
    marker=dict(color=WEAK_RED),
    text=[f'{v:.1f}%' for v in weak_equity['pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

fig4.update_layout(
    title=dict(
        text='<b>Raise Frequency by Hand Strength: Top 10% vs Bottom 10%</b><br><sup>% of actions that are raises within each equity bucket</sup>',
        font=dict(color='black', size=16)
    ),
    xaxis_title='Hand Equity Bucket',
    yaxis_title='Raise % (within bucket)',
    barmode='group',
    height=500,
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(bgcolor='white', font=dict(color='black', size=12)),
    xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=11), tickangle=-20),
    yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
)

fig4.show()

In [ ]:
# ============================================================================
# VISUALIZATION 5: Fold Frequency by Hand Strength (Top vs Bottom Decile)
# ============================================================================
print("\n   Creating fold discipline chart...")

fold_by_equity = hand_strength_stats[hand_strength_stats['action_type'] == 'fold'].copy()
fold_by_equity['equity_bucket'] = pd.Categorical(fold_by_equity['equity_bucket'], categories=bucket_order, ordered=True)
fold_by_equity = fold_by_equity.sort_values('equity_bucket')

# Get data for each group
elite_fold = fold_by_equity[fold_by_equity['comparison_group'] == 'Elite (Top 10%)'].sort_values('equity_bucket')
weak_fold = fold_by_equity[fold_by_equity['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('equity_bucket')

fig5 = go.Figure()

# Add Elite bars (GREEN)
fig5.add_trace(go.Bar(
    name='Elite (Top 10%)',
    x=elite_fold['equity_bucket'].tolist(),
    y=elite_fold['pct'].tolist(),
    marker=dict(color=ELITE_GREEN),
    text=[f'{v:.1f}%' for v in elite_fold['pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

# Add Weakest bars (RED)
fig5.add_trace(go.Bar(
    name='Weakest (Bottom 10%)',
    x=weak_fold['equity_bucket'].tolist(),
    y=weak_fold['pct'].tolist(),
    marker=dict(color=WEAK_RED),
    text=[f'{v:.1f}%' for v in weak_fold['pct']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

fig5.update_layout(
    title=dict(
        text='<b>Fold Frequency by Hand Strength: Top 10% vs Bottom 10%</b><br><sup>% of actions that are folds within each equity bucket</sup>',
        font=dict(color='black', size=16)
    ),
    xaxis_title='Hand Equity Bucket',
    yaxis_title='Fold % (within bucket)',
    barmode='group',
    height=500,
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    legend=dict(bgcolor='white', font=dict(color='black', size=12)),
    xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=11), tickangle=-20),
    yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
)

fig5.show()

In [ ]:
# ============================================================================
# VISUALIZATION 6: Bet Sizing Analysis (SPR-based) - Top vs Bottom Decile
# ============================================================================
print("\n   Creating bet sizing / SPR analysis chart...")

if 'spr' in df_extremes.columns:
    spr_stats = df_extremes.withColumn(
        'spr_bucket',
        F.when(F.col('spr') < 3, 'Low SPR (<3)')
        .when(F.col('spr') < 7, 'Medium SPR (3-7)')
        .when(F.col('spr') < 15, 'High SPR (7-15)')
        .otherwise('Very Deep (>15)')
    ).groupBy('comparison_group', 'spr_bucket', 'action_type').agg(
        F.count('*').alias('count')
    ).toPandas()
    
    spr_stats['total'] = spr_stats.groupby(['comparison_group', 'spr_bucket'])['count'].transform('sum')
    spr_stats['pct'] = spr_stats['count'] / spr_stats['total'] * 100
    
    raise_by_spr = spr_stats[spr_stats['action_type'] == 'bet_or_raise_to'].copy()
    spr_order = ['Low SPR (<3)', 'Medium SPR (3-7)', 'High SPR (7-15)', 'Very Deep (>15)']
    raise_by_spr['spr_bucket'] = pd.Categorical(raise_by_spr['spr_bucket'], categories=spr_order, ordered=True)
    raise_by_spr = raise_by_spr.sort_values('spr_bucket')
    
    # Get data for each group
    elite_spr = raise_by_spr[raise_by_spr['comparison_group'] == 'Elite (Top 10%)'].sort_values('spr_bucket')
    weak_spr = raise_by_spr[raise_by_spr['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('spr_bucket')
    
    fig6 = go.Figure()
    
    # Add Elite bars (GREEN)
    fig6.add_trace(go.Bar(
        name='Elite (Top 10%)',
        x=elite_spr['spr_bucket'].tolist(),
        y=elite_spr['pct'].tolist(),
        marker=dict(color=ELITE_GREEN),
        text=[f'{v:.1f}%' for v in elite_spr['pct']],
        textposition='outside',
        textfont=dict(color='black', size=11)
    ))
    
    # Add Weakest bars (RED)
    fig6.add_trace(go.Bar(
        name='Weakest (Bottom 10%)',
        x=weak_spr['spr_bucket'].tolist(),
        y=weak_spr['pct'].tolist(),
        marker=dict(color=WEAK_RED),
        text=[f'{v:.1f}%' for v in weak_spr['pct']],
        textposition='outside',
        textfont=dict(color='black', size=11)
    ))
    
    fig6.update_layout(
        title=dict(
            text='<b>Raise Frequency by SPR: Top 10% vs Bottom 10%</b><br><sup>How does stack depth affect aggression?</sup>',
            font=dict(color='black', size=16)
        ),
        xaxis_title='SPR Category',
        yaxis_title='Raise %',
        barmode='group',
        height=500,
        template='plotly_white',
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(color='black'),
        legend=dict(bgcolor='white', font=dict(color='black', size=12)),
        xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12)),
        yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
    )
    
    fig6.show()
else:
    print("   SPR column not available, skipping SPR analysis")

In [ ]:
# ============================================================================
# VISUALIZATION 7: Pot Odds Response Analysis (Top vs Bottom Decile)
# ============================================================================
print("\n   Creating pot odds response chart...")

if 'pot_odds_call' in df_extremes.columns:
    pot_odds_stats = df_extremes.filter(
        F.col('pot_odds_call').isNotNull() & (F.col('pot_odds_call') > 0)
    ).withColumn(
        'pot_odds_bucket',
        F.when(F.col('pot_odds_call') < 0.2, 'Great Odds (<20%)')
        .when(F.col('pot_odds_call') < 0.33, 'Good Odds (20-33%)')
        .when(F.col('pot_odds_call') < 0.5, 'Marginal (33-50%)')
        .otherwise('Bad Odds (>50%)')
    ).groupBy('comparison_group', 'pot_odds_bucket', 'action_type').agg(
        F.count('*').alias('count')
    ).toPandas()
    
    pot_odds_stats['total'] = pot_odds_stats.groupby(['comparison_group', 'pot_odds_bucket'])['count'].transform('sum')
    pot_odds_stats['pct'] = pot_odds_stats['count'] / pot_odds_stats['total'] * 100
    
    call_by_odds = pot_odds_stats[pot_odds_stats['action_type'] == 'call_or_check'].copy()
    odds_order = ['Great Odds (<20%)', 'Good Odds (20-33%)', 'Marginal (33-50%)', 'Bad Odds (>50%)']
    call_by_odds['pot_odds_bucket'] = pd.Categorical(call_by_odds['pot_odds_bucket'], categories=odds_order, ordered=True)
    call_by_odds = call_by_odds.sort_values('pot_odds_bucket')
    
    # Get data for each group
    elite_odds = call_by_odds[call_by_odds['comparison_group'] == 'Elite (Top 10%)'].sort_values('pot_odds_bucket')
    weak_odds = call_by_odds[call_by_odds['comparison_group'] == 'Weakest (Bottom 10%)'].sort_values('pot_odds_bucket')
    
    fig7 = go.Figure()
    
    # Add Elite bars (GREEN)
    fig7.add_trace(go.Bar(
        name='Elite (Top 10%)',
        x=elite_odds['pot_odds_bucket'].tolist(),
        y=elite_odds['pct'].tolist(),
        marker=dict(color=ELITE_GREEN),
        text=[f'{v:.1f}%' for v in elite_odds['pct']],
        textposition='outside',
        textfont=dict(color='black', size=11)
    ))
    
    # Add Weakest bars (RED)
    fig7.add_trace(go.Bar(
        name='Weakest (Bottom 10%)',
        x=weak_odds['pot_odds_bucket'].tolist(),
        y=weak_odds['pct'].tolist(),
        marker=dict(color=WEAK_RED),
        text=[f'{v:.1f}%' for v in weak_odds['pct']],
        textposition='outside',
        textfont=dict(color='black', size=11)
    ))
    
    fig7.update_layout(
        title=dict(
            text='<b>Call Frequency by Pot Odds: Top 10% vs Bottom 10%</b><br><sup>How well do they respond to pot odds?</sup>',
            font=dict(color='black', size=16)
        ),
        xaxis_title='Pot Odds Category',
        yaxis_title='Call %',
        barmode='group',
        height=500,
        template='plotly_white',
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(color='black'),
        legend=dict(bgcolor='white', font=dict(color='black', size=12)),
        xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12)),
        yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
    )
    
    fig7.show()
else:
    print("   pot_odds_call column not available, skipping pot odds analysis")

In [ ]:
# ============================================================================
# VISUALIZATION 8: Key Metrics Summary Dashboard (Top vs Bottom Decile)
# ============================================================================
print("\n   Creating summary dashboard...")

# Calculate key metrics for top 10% vs bottom 10%
summary_metrics = df_extremes.groupBy('comparison_group').agg(
    # Aggression
    (F.sum(F.when(F.col('action_type') == 'bet_or_raise_to', 1).otherwise(0)) / 
     F.sum(F.when(F.col('action_type').isin(['bet_or_raise_to', 'call_or_check']), 1).otherwise(0)) * 100
    ).alias('aggression_pct'),
    
    # Fold rate
    (F.sum(F.when(F.col('action_type') == 'fold', 1).otherwise(0)) / F.count('*') * 100
    ).alias('fold_rate'),
    
    # Average hand equity when raising
    F.avg(F.when(F.col('action_type') == 'bet_or_raise_to', F.col('hand_equity'))).alias('avg_equity_when_raising'),
    
    # Average hand equity when calling
    F.avg(F.when(F.col('action_type') == 'call_or_check', F.col('hand_equity'))).alias('avg_equity_when_calling'),
    
    # Average hand equity when folding
    F.avg(F.when(F.col('action_type') == 'fold', F.col('hand_equity'))).alias('avg_equity_when_folding'),
    
    # VPIP proxy
    (F.sum(F.when((F.col('street') == 'preflop') & (F.col('action_type') != 'fold'), 1).otherwise(0)) /
     F.sum(F.when(F.col('street') == 'preflop', 1).otherwise(0)) * 100
    ).alias('vpip_pct'),
    
    # Count
    F.count('*').alias('total_actions')
).toPandas()

# Create comparison table
metrics_list = [
    ('Aggression %', 'aggression_pct', 'Higher = more aggressive'),
    ('Fold Rate %', 'fold_rate', 'Folding frequency'),
    ('VPIP % (preflop)', 'vpip_pct', 'Voluntary put $ in pot'),
    ('Avg Equity When Raising', 'avg_equity_when_raising', 'Hand strength when betting'),
    ('Avg Equity When Calling', 'avg_equity_when_calling', 'Hand strength when calling'),
    ('Avg Equity When Folding', 'avg_equity_when_folding', 'Hand strength when folding'),
]

# Build comparison dataframe
comparison_data = []
for metric_name, col_name, description in metrics_list:
    winner_row = summary_metrics[summary_metrics['comparison_group'] == 'Elite (Top 10%)']
    loser_row = summary_metrics[summary_metrics['comparison_group'] == 'Weakest (Bottom 10%)']
    
    if len(winner_row) > 0 and len(loser_row) > 0:
        winner_val = winner_row[col_name].values[0]
        loser_val = loser_row[col_name].values[0]
        diff = winner_val - loser_val
        comparison_data.append({
            'Metric': metric_name,
            'Elite (Top 10%)': winner_val,
            'Weakest (Bottom 10%)': loser_val,
            'Difference': diff,
            'Description': description
        })

comparison_df = pd.DataFrame(comparison_data)

# Create bar chart of differences with green/red colors
fig8 = go.Figure()

# Use green for positive differences, red for negative
bar_colors = [ELITE_GREEN if x > 0 else WEAK_RED for x in comparison_df['Difference']]

fig8.add_trace(go.Bar(
    x=comparison_df['Metric'].tolist(),
    y=comparison_df['Difference'].tolist(),
    marker=dict(color=bar_colors),
    text=[f'{x:+.2f}' for x in comparison_df['Difference']],
    textposition='outside',
    textfont=dict(color='black', size=11)
))

fig8.update_layout(
    title=dict(
        text='<b>Top 10% vs Bottom 10%: Key Metric Differences</b><br><sup>Green = Elite do MORE, Red = Elite do LESS</sup>',
        font=dict(color='black', size=16)
    ),
    xaxis_title='Metric',
    yaxis_title='Difference (Elite - Weakest)',
    height=500,
    showlegend=False,
    template='plotly_white',
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(color='black'),
    xaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=11)),
    yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', tickfont=dict(color='black', size=12))
)
fig8.add_hline(y=0, line_dash="dash", line_color="gray")

fig8.show()

# Print the comparison table
print("\n   " + "=" * 95)
print(f"   {'Metric':<30} {'Elite (Top 10%)':>18} {'Weakest (Bot 10%)':>18} {'Difference':>15}")
print("   " + "-" * 95)
for _, row in comparison_df.iterrows():
    diff_str = f"{row['Difference']:+.2f}"
    print(f"   {row['Metric']:<30} {row['Elite (Top 10%)']:>18.2f} {row['Weakest (Bottom 10%)']:>18.2f} {diff_str:>15}")
print("   " + "=" * 95)

print("\n   ✓ All visualizations created successfully!")

In [ ]:
# Final Summary
elapsed = time.time() - start_time

print("\n" + "=" * 80)
print("SP-8: Player Performance Analysis COMPLETE!")
print("=" * 80)
print(f"\nRuntime: {elapsed:.1f} seconds")
print(f"Output: {OUTPUT_PATH}")
print(f"Dataset: {final_count:,} rows, {final_cols} columns")

print(f"\n" + "-" * 80)
print("KEY METRICS:")
print("-" * 80)
print(f"   Total players analyzed: {total_players:,}")
print(f"   Players with >= {MIN_HANDS_FOR_EVALUATION} hands: {players_with_enough:,}")
print(f"   Elite players (top 10%): {winning_count:,}")
print(f"   Top decile threshold: >= {top_decile_threshold:.2f} BB/100")

print(f"\n" + "-" * 80)
print("DECILE-BASED CLASSIFICATION:")
print("-" * 80)
print("   Decile 10 (top 10%):   Elite - is_winning_player = 1")
print("   Decile 9 (top 20%):    Strong")
print("   Decile 7-8 (top 40%):  Above Average")
print("   Decile 5-6 (40-60%):   Average")
print("   Decile 3-4 (bot 40%):  Below Average")
print("   Decile 1-2 (bot 20%):  Weak")
print("   Decile 0:              Insufficient data")

print(f"\n" + "-" * 80)
print("NEW COLUMNS ADDED:")
print("-" * 80)
print("   performance_decile   - 1-10 ranking (10 = best)")
print("   is_winning_player    - 1 if top decile (10), 0 otherwise")
print("   player_category      - Descriptive category based on decile")
print("   profit_per_100_hands - Player's profit rate (BB per 100 hands)")
print("   win_rate             - Fraction of hands player won")
print("   total_hands          - Number of hands player has in dataset")

print(f"\n" + "-" * 80)
print("DATA LINEAGE:")
print("-" * 80)
print(f"   Input: SP-7 output ({SP7_INPUT_PATH})")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Player stats: {PLAYER_STATS_PATH}")

print(f"\n" + "-" * 80)
print("TRACE SUMMARY:")
print("-" * 80)
print(f"   TRACE_HAND_ID: {TRACE_HAND_ID}")

print(f"\n[SUCCESS] Player performance analysis complete!")
print(f"Next step: Run SP-9 (09_PolicyTraining) to train on TOP DECILE player actions")